# Compute contribution decay from PostgreSQL, inspect it, then write to PostgreSQL

This notebook deliberately separates algorithm computation from database writes:

1. Compute decay results and save bounded Parquet parts.
2. Inspect and manually verify those results.
3. Run a separate explicit cell to replace and write the verified results to the configured PostgreSQL score schema.


In [31]:
# Configuration
from datetime import datetime, timezone
from pathlib import Path

from mindshare_config import pg_score_schema

SCOPE = "project"  # "project" or "global"
PROJECT_KEYWORD = "quipnetwork"  # only used for project scope
SOURCE_TARGET = "pg"
DESTINATION_SCHEMA = pg_score_schema()
WRITE_METHOD = "copy"  # fastest PostgreSQL benchmark; alternative: "multirow"
SOURCE_BATCH_SIZE = 100_000
WRITE_BATCH_SIZE = 100_000
INCLUDE_ACTIVE_MULTIPLIERS = False

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = Path("output") / f"{SCOPE}_{PROJECT_KEYWORD}_{RUN_ID}"

assert SCOPE in {"project", "global"}
assert SOURCE_TARGET == "pg"
assert DESTINATION_SCHEMA != "mindshare"
if SCOPE == "project":
    assert PROJECT_KEYWORD

print("Computed results will be stored in:", OUTPUT_DIR)


Computed results will be stored in: output/project_quipnetwork_20260615T061427Z


In [32]:
# Imports
from importlib import reload
from inspect import getsource
from time import perf_counter

import polars as pl
import mindshare_compute.db as db_module

# Reload because Jupyter caches imported modules between runs.
reload(db_module)

from mindshare_compute.decay import DecayComputer

iter_decay_source = db_module.iter_decay_source
DecayResultWriter = db_module.DecayResultWriter

pl.Config.set_tbl_rows(20)
assert "destination schema must not be the read-only source schema" in getsource(DecayResultWriter.__init__)
assert "with cursor.copy(copy_statement) as copy" in getsource(DecayResultWriter._write_batch_copy)
print("Loaded PostgreSQL-safe writer from:", db_module.__file__)


Loaded PostgreSQL-safe writer from: /home/babin411/Nucleus/mindshare-backend-optimization/mindshare_compute/db.py


## 1. Compute algorithm results only

This cell reads only from PostgreSQL `mindshare`. It **does not write to any database**. Each computed batch is saved as a Parquet part so the complete result can be inspected without holding it all in WSL memory.


In [33]:
computer = DecayComputer(
    SCOPE,
    include_active_multipliers=INCLUDE_ACTIVE_MULTIPLIERS,
)
source = iter_decay_source(
    SCOPE,
    PROJECT_KEYWORD if SCOPE == "project" else None,
    target=SOURCE_TARGET,
    batch_size=SOURCE_BATCH_SIZE,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
compute_start = perf_counter()
compute_seconds = 0.0
rows_computed = 0
parts_written = 0
last_computed_batch = None

for batch_number, source_batch in enumerate(source):
    started = perf_counter()
    result_batch = computer.process_batch(source_batch)
    compute_seconds += perf_counter() - started

    result_batch.write_parquet(OUTPUT_DIR / f"part-{batch_number:05d}.parquet")
    rows_computed += result_batch.height
    parts_written += 1
    last_computed_batch = result_batch
    print(f"computed part={batch_number:05d} total_rows={rows_computed:,}")

compute_summary = pl.DataFrame({
    "source_target": [SOURCE_TARGET],
    "source_schema": [db_module.source_schema(SOURCE_TARGET)],
    "scope": [SCOPE],
    "project_keyword": [PROJECT_KEYWORD if SCOPE == "project" else None],
    "rows_computed": [rows_computed],
    "parquet_parts": [parts_written],
    "algorithm_compute_seconds": [compute_seconds],
    "compute_and_parquet_wall_seconds": [perf_counter() - compute_start],
})
compute_summary


computed part=00000 total_rows=100,000
computed part=00001 total_rows=200,000
computed part=00002 total_rows=300,000
computed part=00003 total_rows=400,000
computed part=00004 total_rows=500,000
computed part=00005 total_rows=600,000
computed part=00006 total_rows=700,000
computed part=00007 total_rows=800,000
computed part=00008 total_rows=900,000
computed part=00009 total_rows=1,000,000
computed part=00010 total_rows=1,100,000
computed part=00011 total_rows=1,200,000
computed part=00012 total_rows=1,300,000
computed part=00013 total_rows=1,400,000
computed part=00014 total_rows=1,500,000
computed part=00015 total_rows=1,600,000
computed part=00016 total_rows=1,700,000
computed part=00017 total_rows=1,800,000
computed part=00018 total_rows=1,900,000
computed part=00019 total_rows=2,000,000
computed part=00020 total_rows=2,100,000
computed part=00021 total_rows=2,200,000
computed part=00022 total_rows=2,300,000
computed part=00023 total_rows=2,400,000
computed part=00024 total_rows=2,4

source_target,source_schema,scope,project_keyword,rows_computed,parquet_parts,algorithm_compute_seconds,compute_and_parquet_wall_seconds
str,str,str,str,i64,i64,f64,f64
"""pg""","""mindshare""","""project""","""quipnetwork""",2450876,25,24.343458,383.11891


## 2. Manually inspect and verify

These cells scan the saved Parquet files lazily. Nothing is written to PostgreSQL here. Add any manual checks you need before running the database-write section.


In [34]:
result_scan = pl.scan_parquet(OUTPUT_DIR / "*.parquet")

result_scan.head(50).collect()


project_keyword,reply_post_id,original_post_id,replier_x_id,original_author_x_id,post_created_at,replier_base_score,effective_score,contribution_score,reply_number,local_reply_count,decay_type
str,str,str,str,str,"datetime[μs, UTC]",f64,f64,f64,i64,i64,str
"""quipnetwork""","""2044210856461062512""","""2044202354291904575""","""""","""2285666115""",2026-04-15 00:27:02 UTC,0.0,0.0,0.0,1,1,"""FIRST_REPLY"""
"""quipnetwork""","""2044211242013712840""","""2043365730612301879""","""""","""1437682537149526019""",2026-04-15 00:28:34 UTC,0.0,0.0,0.0,2,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044212093352849894""","""2044210505779425534""","""""","""""",2026-04-15 00:31:57 UTC,0.0,0.0,0.0,3,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044212454591475833""","""2044212315600629869""","""""","""1719270086543040512""",2026-04-15 00:33:23 UTC,0.0,0.0,0.0,4,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044212538334724557""","""2042783412885557584""","""""","""931391403350884353""",2026-04-15 00:33:43 UTC,0.0,0.0,0.0,5,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044212581225607305""","""2042757777379201470""","""""","""1619588940775960576""",2026-04-15 00:33:53 UTC,0.0,0.0,0.0,6,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044219829708828956""","""2044219077045170329""","""""","""896279370""",2026-04-15 01:02:41 UTC,0.0,0.0,0.0,7,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044219829641720237""","""2044213824837038160""","""""","""1482396650471661572""",2026-04-15 01:02:41 UTC,0.0,0.0,0.0,8,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044220059959402861""","""2044219077045170329""","""""","""896279370""",2026-04-15 01:03:36 UTC,0.0,0.0,0.0,9,2,"""LOCAL_DECAY"""


In [35]:
# Useful high-level verification before writing.
result_scan.group_by("decay_type").agg(
    pl.len().alias("rows"),
    pl.col("contribution_score").min().alias("min_score"),
    pl.col("contribution_score").mean().alias("mean_score"),
    pl.col("contribution_score").max().alias("max_score"),
).sort("decay_type").collect()


decay_type,rows,min_score,mean_score,max_score
str,u32,f64,f64,f64
"""FIRST_REPLY""",41495,0.0,75.088747,2645.09
"""GLOBAL_DECAY""",575436,0.0,14.041813,2372.19
"""LOCAL_DECAY""",1833945,0.0,2.84168,1019.65


In [36]:
# Change these filters for a replier or author you want to verify manually.
VERIFY_REPLIER_X_ID = None
VERIFY_ORIGINAL_AUTHOR_X_ID = None

verification = result_scan
if VERIFY_REPLIER_X_ID:
    verification = verification.filter(pl.col("replier_x_id") == VERIFY_REPLIER_X_ID)
if VERIFY_ORIGINAL_AUTHOR_X_ID:
    verification = verification.filter(
        pl.col("original_author_x_id") == VERIFY_ORIGINAL_AUTHOR_X_ID
    )

verification.sort("replier_x_id", "post_created_at").head(200).collect()


project_keyword,reply_post_id,original_post_id,replier_x_id,original_author_x_id,post_created_at,replier_base_score,effective_score,contribution_score,reply_number,local_reply_count,decay_type
str,str,str,str,str,"datetime[μs, UTC]",f64,f64,f64,i64,i64,str
"""quipnetwork""","""2044210856461062512""","""2044202354291904575""","""""","""2285666115""",2026-04-15 00:27:02 UTC,0.0,0.0,0.0,1,1,"""FIRST_REPLY"""
"""quipnetwork""","""2044211242013712840""","""2043365730612301879""","""""","""1437682537149526019""",2026-04-15 00:28:34 UTC,0.0,0.0,0.0,2,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044212093352849894""","""2044210505779425534""","""""","""""",2026-04-15 00:31:57 UTC,0.0,0.0,0.0,3,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044212454591475833""","""2044212315600629869""","""""","""1719270086543040512""",2026-04-15 00:33:23 UTC,0.0,0.0,0.0,4,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044212538334724557""","""2042783412885557584""","""""","""931391403350884353""",2026-04-15 00:33:43 UTC,0.0,0.0,0.0,5,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044212581225607305""","""2042757777379201470""","""""","""1619588940775960576""",2026-04-15 00:33:53 UTC,0.0,0.0,0.0,6,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044219829708828956""","""2044219077045170329""","""""","""896279370""",2026-04-15 01:02:41 UTC,0.0,0.0,0.0,7,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044219829641720237""","""2044213824837038160""","""""","""1482396650471661572""",2026-04-15 01:02:41 UTC,0.0,0.0,0.0,8,1,"""GLOBAL_DECAY"""
"""quipnetwork""","""2044220059959402861""","""2044219077045170329""","""""","""896279370""",2026-04-15 01:03:36 UTC,0.0,0.0,0.0,9,2,"""LOCAL_DECAY"""


## 3. Write verified Parquet results to PostgreSQL

Run this cell only after manually verifying the algorithm output. It writes only to the configured PostgreSQL score schema; the source `mindshare` schema remains read-only.

- Project scope deletes that project's existing rows from the configured score schema's `test_contribution_scores` table before writing.
- Global scope truncates the configured score schema's `test_global_contribution_scores` table before writing.
- The writer creates the schema/table when they do not exist.


In [37]:
# Explicit safety switch: change to True only after inspecting the results above.
WRITE_VERIFIED_RESULTS_TO_POSTGRES = True

if not WRITE_VERIFIED_RESULTS_TO_POSTGRES:
    raise RuntimeError(
        "Set WRITE_VERIFIED_RESULTS_TO_POSTGRES = True after manually verifying the Parquet results."
    )

write_start = perf_counter()
write_seconds = 0.0
rows_written = 0

# collect_batches streams the verified Parquet dataset without loading it all.
parquet_batches = pl.scan_parquet(OUTPUT_DIR / "*.parquet").collect_batches(
    chunk_size=WRITE_BATCH_SIZE,
    maintain_order=True,
)

with DecayResultWriter(
    SCOPE,
    PROJECT_KEYWORD if SCOPE == "project" else None,
    target="pg",
    destination_schema=DESTINATION_SCHEMA,
    write_method=WRITE_METHOD,
) as writer:
    for batch_number, result_batch in enumerate(parquet_batches):
        started = perf_counter()
        rows_written += writer.write_batch(result_batch)
        write_seconds += perf_counter() - started
        print(f"wrote batch={batch_number:05d} total_rows={rows_written:,}")

write_summary = pl.DataFrame({
    "write_target": ["pg"],
    "destination_schema": [DESTINATION_SCHEMA],
    "write_method": [WRITE_METHOD],
    "rows_written": [rows_written],
    "postgres_write_seconds": [write_seconds],
    "write_wall_seconds": [perf_counter() - write_start],
})
write_summary


wrote batch=00000 total_rows=100,000
wrote batch=00001 total_rows=200,000
wrote batch=00002 total_rows=300,000
wrote batch=00003 total_rows=400,000
wrote batch=00004 total_rows=500,000
wrote batch=00005 total_rows=600,000
wrote batch=00006 total_rows=700,000
wrote batch=00007 total_rows=800,000
wrote batch=00008 total_rows=900,000
wrote batch=00009 total_rows=1,000,000
wrote batch=00010 total_rows=1,100,000
wrote batch=00011 total_rows=1,200,000
wrote batch=00012 total_rows=1,300,000
wrote batch=00013 total_rows=1,400,000
wrote batch=00014 total_rows=1,500,000
wrote batch=00015 total_rows=1,600,000
wrote batch=00016 total_rows=1,700,000
wrote batch=00017 total_rows=1,800,000
wrote batch=00018 total_rows=1,900,000
wrote batch=00019 total_rows=2,000,000
wrote batch=00020 total_rows=2,100,000
wrote batch=00021 total_rows=2,200,000
wrote batch=00022 total_rows=2,300,000
wrote batch=00023 total_rows=2,400,000
wrote batch=00024 total_rows=2,450,876


write_target,destination_schema,write_method,rows_written,postgres_write_seconds,write_wall_seconds
str,str,str,i64,f64,f64
"""pg""","""configured_score_schema""","""copy""",2450876,103.54214,110.141701
